# Bench and MeasurementSession Integration Example

Open a bench, run a two-parameter sweep, and inspect the resulting data. The
bench is forced to use the built-in power-supply simulator. The transistor current
is a deterministic illustration, not a circuit simulation or real measurement.

Requires Python 3.11 or newer and PyTestLab (`python -m pip install pytestlab`).

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from pytestlab import Bench, MeasurementSession

## Define the experiment

The bench supplies its power supply and local experiment metadata to the session.
The local database lives in a temporary directory and is removed after the run.

In [ ]:
def run_bench_session():
    with TemporaryDirectory(prefix="pytestlab-bench-session-") as scratch:
        database = Path(scratch) / "transistor_measurements.db"
        config = {
            "bench_name": "Simulated transistor bench",
            "simulate": True,
            "instruments": {"psu": {"profile": "keysight/EDU36311A"}},
            "experiment": {
                "title": "Transistor sweep example",
                "description": "Illustrative voltage sweep with a deterministic current model",
                "database_path": str(database),
            },
        }
        with Bench.open(config, force_simulation=True) as bench:
            with MeasurementSession(bench=bench, compliance=False) as session:
                session.parameter("base_v", np.linspace(0.6, 1.0, 5), unit="V")
                session.parameter("collector_v", np.linspace(0.0, 5.0, 10), unit="V")

                @session.acquire
                def acquire(base_v, collector_v, psu):
                    psu.set_current(1, 0.05)
                    psu.set_current(2, 0.5)
                    psu.set_voltage(1, float(base_v))
                    psu.set_voltage(2, float(collector_v))
                    try:
                        psu.output(1, True)
                        psu.output(2, True)
                        # Illustrative only: this is not a measured transistor current.
                        return {"modeled_current_a": 0.02 * max(base_v - 0.6, 0) * collector_v}
                    finally:
                        psu.output(1, False)
                        psu.output(2, False)

                experiment = session.run(show_progress=False)

            assert bench.db is not None and database.is_file()
            assert experiment.data.height == 50
            print(f"Collected {experiment.data.height} simulated points using {database.name}.")
            return experiment

## Run the sweep

In [ ]:
experiment = run_bench_session()

## Inspect the first rows

In [ ]:
experiment.data.head(10)